In [ ]:
!pip install transformers datasets librosa torch torchaudio opensmile praat-parselmouth

###Φόρτωση και speaker-independent διαχωρισμός

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        return "OAF" if "OAF" in filename else ("YAF" if "YAF" in filename else "unknown")

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()

labels = sorted(set(dataset["train"]["emotion"]))
num_labels = len(labels)
label2id_fixed = {str(label): i for i, label in enumerate(labels)}
id2label_fixed = {i: str(label) for i, label in enumerate(labels)}
print("Κλάσεις:", labels)

###Φιλτράρισμα σχεδόν-σιωπηλών δειγμάτων

In [ ]:
import numpy as np

def is_valid_audio(batch):
    audio = batch["audio"]["array"]
    if len(audio) == 0:
        return False
    return np.std(audio) >= 1e-4

print("Πριν:", len(dataset["train"]), len(dataset["test"]))
dataset["train"] = dataset["train"].filter(is_valid_audio, load_from_cache_file=False)
dataset["test"] = dataset["test"].filter(is_valid_audio, load_from_cache_file=False)
print("Μετά:", len(dataset["train"]), len(dataset["test"]))

###Φόρτωση μοντέλου

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

checkpoint = "facebook/wav2vec2-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)
model = AutoModelForAudioClassification.from_pretrained(
    checkpoint, num_labels=num_labels, label2id=label2id_fixed, id2label=id2label_fixed,
    ignore_mismatched_sizes=True,
).to("cuda")
model.freeze_feature_encoder()

###Φόρτωση μοντέλου

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = label2id_fixed[batch["emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset["train"].column_names, load_from_cache_file=False)

###Ορισμός μετρικών

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
    average_precision_score, log_loss, confusion_matrix,
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    cm = confusion_matrix(labels_true, predictions)
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

###Δημιουργία data collator

In [ ]:
class DataCollatorCTCWithPadding:
    def __init__(self, feature_extractor, padding=True):
        self.feature_extractor = feature_extractor
        self.padding = padding

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        batch = self.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

###Ορισμός υπερπαραμέτρων και εκπαίδευση

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned-TESS",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    fp16=False,
    learning_rate=3e-5,
    warmup_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιοργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δηνιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/wav2vec-finetuned-TESS"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")